# Smart Grocery Expiry and Discount Advisor — Dataset-Based ML Practical
**Dataset used:** `GroceryDataset.csv` (1,757 product-catalog rows, 8 original columns).

## Important dataset limitation
This dataset contains product category, listed price, discount text, rating text, title, currency, feature text and product description. It does **not** contain daily sales, current stock, expiry dates, dates of sale or actual unsold/waste outcomes. Therefore, it cannot honestly train an expiry/waste-risk classifier or a demand-forecasting model. This notebook uses the real dataset for EDA and demonstrates:
1. Price estimation with Linear Regression and an ANN regressor.
2. Discount-status classification with Decision Tree, KNN and Naive Bayes.
These are dataset-based demonstrations, not substitutes for the original expiry/waste models. To implement real expiry alerts, collect the missing inventory and sales fields.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.base import clone
import joblib

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Libraries imported successfully!")

## 2. Load the Actual Dataset

In [ ]:
# Put GroceryDataset.csv in the same folder as this notebook.
df = pd.read_csv("GroceryDataset.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

## 3. Initial Data Inspection

In [ ]:
df.info()
display(df.describe(include="all").T)
print("\nMissing values:")
display(df.isnull().sum().to_frame("missing_count"))
print("\nDuplicate rows:", df.duplicated().sum())
print("\nNumber of product categories:", df["Sub Category"].nunique())
print("\nMost common discount labels:")
display(df["Discount"].value_counts().head(15))

## 4. Data Cleaning and Feature Engineering

In [ ]:
data = df.copy()
data = data.drop_duplicates().reset_index(drop=True)

# Convert price strings such as '$56.99' into numeric values.
data["Price_numeric"] = pd.to_numeric(
    data["Price"].astype(str).str.replace(r"[^0-9.]", "", regex=True),
    errors="coerce"
)

# Rating text often looks like: 'Rated 4.3 out of 5 stars based on 265 reviews.'
data["Rating_numeric"] = pd.to_numeric(
    data["Rating"].astype(str).str.extract(r"Rated\s*([0-5](?:\.\d+)?)", expand=False),
    errors="coerce"
)
data["Review_count"] = pd.to_numeric(
    data["Rating"].astype(str).str.extract(r"based on\s*(\d+)\s*reviews?", expand=False),
    errors="coerce"
)

data["Feature"] = data["Feature"].fillna("")
data["Product Description"] = data["Product Description"].fillna("")
data["Sub Category"] = data["Sub Category"].fillna("Unknown")
data["Discount"] = data["Discount"].fillna("Unknown")
data["Title"] = data["Title"].fillna("")

data["Description_length"] = data["Product Description"].str.len()
data["Feature_count"] = data["Feature"].apply(
    lambda x: sum(1 for line in str(x).splitlines() if line.strip())
)
data["Has_discount"] = (data["Discount"].str.lower() != "no discount").astype(int)

print("Cleaned shape:", data.shape)
display(data[["Sub Category", "Price_numeric", "Discount", "Rating_numeric",
              "Review_count", "Description_length", "Feature_count"]].head())
print("\nMissing engineered values:")
display(data[["Price_numeric", "Rating_numeric", "Review_count"]].isna().sum())

## 5. EDA — Visual Analysis

In [ ]:
plt.figure(figsize=(10,5))
data["Sub Category"].value_counts().head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Product Sub-Categories")
plt.xlabel("Number of products")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
data["Price_numeric"].dropna().clip(upper=data["Price_numeric"].quantile(0.98)).hist(bins=30, edgecolor="black")
plt.title("Product Price Distribution (clipped at 98th percentile for readability)")
plt.xlabel("Price")
plt.ylabel("Product count")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
data["Has_discount"].value_counts().rename(index={0:"No discount",1:"Discount listed"}).plot(kind="bar")
plt.title("Products With and Without a Listed Discount")
plt.ylabel("Product count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
data["Rating_numeric"].dropna().hist(bins=20, edgecolor="black")
plt.title("Customer Rating Distribution (available ratings only)")
plt.xlabel("Rating out of 5")
plt.ylabel("Product count")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=data, x="Review_count", y="Rating_numeric", alpha=0.5)
plt.title("Rating vs Review Count")
plt.xscale("log")
plt.xlabel("Review count (log scale)")
plt.show()

In [ ]:
numeric_cols = ["Price_numeric", "Rating_numeric", "Review_count",
                "Description_length", "Feature_count", "Has_discount"]
plt.figure(figsize=(9,6))
sns.heatmap(data[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Engineered Numeric Features")
plt.tight_layout()
plt.show()

## 6. Prepare Features and Targets
### Model A — Price estimation
Target: `Price_numeric`. Features: category, discount label, rating, review count, description length and feature count. This estimates listed product price from catalog attributes.

### Model B — Discount status
Target: `Has_discount` (0 = No Discount, 1 = any listed discount). This is highly imbalanced in the source dataset, so do not judge it by accuracy alone. Use precision, recall, F1 and the confusion matrix. `Discount` itself is excluded from input features to avoid leaking the answer.

In [ ]:
model_data = data.dropna(subset=["Price_numeric"]).copy()
price_features = ["Sub Category", "Discount", "Rating_numeric", "Review_count",
                  "Description_length", "Feature_count"]
X_price = model_data[price_features]
y_price = model_data["Price_numeric"]

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_price, y_price, test_size=0.2, random_state=RANDOM_STATE
)

cat_price = ["Sub Category", "Discount"]
num_price = ["Rating_numeric", "Review_count", "Description_length", "Feature_count"]
price_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_price),
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_price)
])

X_discount = data[["Sub Category", "Price_numeric", "Rating_numeric", "Review_count",
                   "Description_length", "Feature_count"]].copy()
y_discount = data["Has_discount"]
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_discount, y_discount, test_size=0.2, random_state=RANDOM_STATE, stratify=y_discount
)
discount_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["Sub Category"]),
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]),
     ["Price_numeric", "Rating_numeric", "Review_count", "Description_length", "Feature_count"])
])
print("Price model train/test:", X_train_p.shape, X_test_p.shape)
print("Discount model train/test:", X_train_d.shape, X_test_d.shape)
print("Discount target distribution:")
display(y_discount.value_counts(normalize=True).rename("proportion"))

## 7. Linear Regression — Estimate Listed Product Price

In [ ]:
linear_model = Pipeline([
    ("preprocessor", clone(price_preprocessor)),
    ("model", LinearRegression())
])
linear_model.fit(X_train_p, y_train_p)
price_pred = np.maximum(0, linear_model.predict(X_test_p))
print("MAE :", round(mean_absolute_error(y_test_p, price_pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test_p, price_pred)), 2))
print("R²  :", round(r2_score(y_test_p, price_pred), 3))

plt.figure(figsize=(7,5))
plt.scatter(y_test_p, price_pred, alpha=0.6)
lims = [min(y_test_p.min(), price_pred.min()), max(y_test_p.max(), price_pred.max())]
plt.plot(lims, lims, "r--")
plt.xlabel("Actual listed price")
plt.ylabel("Predicted listed price")
plt.title("Linear Regression: Actual vs Predicted Price")
plt.tight_layout()
plt.show()

## 8. Decision Tree, KNN and Naive Bayes — Discount Status

In [ ]:
discount_models = {
    "Decision Tree": Pipeline([
        ("preprocessor", clone(discount_preprocessor)),
        ("model", DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("preprocessor", clone(discount_preprocessor)),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ]),
    "Naive Bayes": Pipeline([
        ("preprocessor", clone(discount_preprocessor)),
        ("model", MultinomialNB())
    ])
}

# MultinomialNB expects non-negative features. Use a dedicated preprocessing pipeline
# with one-hot categorical features and imputed non-scaled numeric values.
nb_preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["Sub Category"]),
    ("num", SimpleImputer(strategy="median"),
     ["Price_numeric", "Rating_numeric", "Review_count", "Description_length", "Feature_count"])
])
discount_models["Naive Bayes"] = Pipeline([
    ("preprocessor", nb_preprocessor),
    ("model", MultinomialNB())
])

trained_discount_models = {}
metric_rows = []
for name, model in discount_models.items():
    model.fit(X_train_d, y_train_d)
    pred = model.predict(X_test_d)
    trained_discount_models[name] = model
    metric_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_d, pred),
        "Precision (discount)": precision_score(y_test_d, pred, zero_division=0),
        "Recall (discount)": recall_score(y_test_d, pred, zero_division=0),
        "F1 (discount)": f1_score(y_test_d, pred, zero_division=0)
    })
display(pd.DataFrame(metric_rows).sort_values("F1 (discount)", ascending=False))

In [ ]:
for name, model in trained_discount_models.items():
    pred = model.predict(X_test_d)
    print("\n", "="*15, name, "="*15)
    print(classification_report(y_test_d, pred,
          target_names=["No discount", "Discount listed"], zero_division=0))
    cm = confusion_matrix(y_test_d, pred, labels=[0,1])
    ConfusionMatrixDisplay(cm, display_labels=["No discount", "Discount listed"]).plot(
        cmap="Blues", values_format="d"
    )
    plt.title(f"Confusion Matrix — {name}")
    plt.show()

## 9. ANN — Optional Listed Price Estimation

In [ ]:
ann_model = Pipeline([
    ("preprocessor", clone(price_preprocessor)),
    ("model", MLPRegressor(hidden_layer_sizes=(32,16), max_iter=1000,
                           early_stopping=True, random_state=RANDOM_STATE))
])
ann_model.fit(X_train_p, y_train_p)
ann_pred = np.maximum(0, ann_model.predict(X_test_p))
print("ANN MAE :", round(mean_absolute_error(y_test_p, ann_pred), 2))
print("ANN RMSE:", round(np.sqrt(mean_squared_error(y_test_p, ann_pred)), 2))
print("ANN R²  :", round(r2_score(y_test_p, ann_pred), 3))

## 10. Save Trained Models

In [ ]:
joblib.dump(linear_model, "grocery_price_model.joblib")
joblib.dump(trained_discount_models["Decision Tree"], "grocery_discount_status_model.joblib")
joblib.dump(trained_discount_models["KNN"], "grocery_knn_discount_model.joblib")
joblib.dump(trained_discount_models["Naive Bayes"], "grocery_naive_bayes_discount_model.joblib")
joblib.dump(ann_model, "grocery_ann_price_model.joblib")
print("Models saved successfully!")

## 11. Custom Prediction Demo

In [ ]:
demo_string = "category=Bakery & Desserts, discount=No Discount, rating=4.3, reviews=265, description_length=900, feature_count=4"
demo = pd.DataFrame([{
    "Sub Category": "Bakery & Desserts",
    "Discount": "No Discount",
    "Rating_numeric": 4.3,
    "Review_count": 265,
    "Description_length": 900,
    "Feature_count": 4
}])
estimated_price = max(0.0, float(linear_model.predict(demo)[0]))
discount_input = pd.DataFrame([{
    "Sub Category": "Bakery & Desserts",
    "Price_numeric": estimated_price,
    "Rating_numeric": 4.3,
    "Review_count": 265,
    "Description_length": 900,
    "Feature_count": 4
}])
discount_pred = int(trained_discount_models["Decision Tree"].predict(discount_input)[0])
print("Demo input:", demo_string)
print(f"Estimated listed price: ${estimated_price:.2f}")
print("Discount status prediction:", "Discount listed" if discount_pred else "No discount")
print("Note: this predicts catalog patterns, not whether stock will expire or go unsold.")

## 12. Final Analysis and Limitations
- The original dataset is a product catalog, not a time-series inventory dataset.
- The notebook uses its actual columns and derives numeric price, rating, review-count and text-length features.
- Regression target: listed product price.
- Classification target: whether a discount is listed.
- Discount classes are strongly imbalanced; review recall/F1 and confusion matrices.
- There are no sales history, stock counts, expiry dates, or actual waste labels. Therefore, this dataset alone cannot support demand forecasting, expiry alerts or unsold-stock risk prediction.
- To finish the intended expiry advisor, join this catalog to store-level daily records containing product ID, date, opening stock, units sold, expiry date, promotion and units wasted/unsold. Then train demand and waste-risk models on those labels.